# Additional weight-only ECS Jacobians

This notebook adds five actual single-checkpoint derivatives:
the gap-aware hard projector, soft logistic ECS projector, exact
outer trace-free log-Gram, multiscale trace-free resolvent, and
trace-free log Feshbach effective core. Every fitted observation is
a nonzero eigenvalue of $J^*J$, obtained by fitting the Jacobian
amplitudes once and applying the exact amplitude-to-energy change
of variables.

The checkpoint SVD frame is frozen wherever the map is anchored.
In that frame the Feshbach coupling block $B$ is zero, so its shell
terms vanish at first order. That collapse is saved as a result,
not hidden by rotating to an unrelated basis.


In [ ]:
# Papermill parameters. Override these values in an injected cell.
RUN_ROOT = ""
OUTPUT_ROOT = ""
CHECKPOINT_CACHE_ROOT = ""
CONFIG_PATH = ""
PROFILE = "pilot_1000_epochs"
PROTOCOL_SLUG = ""
SEEDS = [1337, 2027, 31415]
CHECKPOINT_PAYLOAD_CACHE_SIZE = 24
SHOW_PLOTS = True
REQUIRE_ARTIFACTS = True
ALLOW_TEMPORARY_LONG_RUN = False
OPTIMIZER_SLUGS = ["adamw", "muon", "muonclip_rms"]
LAYERS = ["fc1.weight"]
MAXIMUM_CHECKPOINTS = 100
ANALYSIS_EPOCH_STRIDE = 1
TOP_K_VALUES = [0, 1, 2, 3, 4, 5]
MINIMUM_TAIL = 8
ECS_RANK_RCOND = 1e-9
SOFT_TEMPERATURE_GAP_RATIOS = [0.25, 0.5, 1.0, 2.0]
RESOLVENT_Z_BOUNDARY_RATIOS = [0.1, 1.0, 10.0]
FESHBACH_Z_SHELL_FLOOR_RATIO = 0.5
METHOD_SLUG = "additional_weight_only_ecs_jacobians"
ANALYSIS_CONTRACT_TOKEN = "additional_weight_only_ecs_jacobians_v1"
PLOT_TITLE = "Additional weight-only Jacobian energies: alpha with 95% seed CI"


In [ ]:
from pathlib import Path
from dataclasses import asdict, is_dataclass
from functools import lru_cache
import inspect
import json
import os
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (cwd, *cwd.parents)
        if (candidate / "baseline" / "rg_baselines").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find baseline/rg_baselines. Launch Jupyter from a clone of "
        "CalculatedContent/rg_optimizers."
    )
BASELINE_ROOT = REPO_ROOT / "baseline"
EXPERIMENT_ROOT = BASELINE_ROOT / "experiments" / "mnist_mlp3_tangent_rg"
if str(BASELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(BASELINE_ROOT))

default_root = os.environ.get(
    "RG_MNIST_TANGENT_ROOT", "/tmp/rg-mnist-mlp3-tangent-rg"
)
RUN_ROOT_PATH = Path(RUN_ROOT or default_root).expanduser().resolve()

default_checkpoint_cache_root = os.environ.get(
    "RG_MNIST_TANGENT_CHECKPOINT_CACHE_ROOT",
    "/tmp/rg-mnist-mlp3-tangent-checkpoints",
)
CHECKPOINT_CACHE_ROOT_PATH = Path(
    CHECKPOINT_CACHE_ROOT or default_checkpoint_cache_root
).expanduser().resolve()

def _suite_name_from_profile():
    if str(PROTOCOL_SLUG).strip():
        return str(PROTOCOL_SLUG).strip()
    candidate = (
        Path(CONFIG_PATH).expanduser()
        if str(CONFIG_PATH).strip()
        else EXPERIMENT_ROOT / "configs" / f"{PROFILE}.yaml"
    )
    if candidate.is_file():
        if candidate.suffix.lower() == ".json":
            payload = json.loads(candidate.read_text(encoding="utf-8"))
            value = payload.get("protocol", {}).get("suite_name")
            if value:
                return str(value)
        else:
            for line in candidate.read_text(encoding="utf-8").splitlines():
                stripped = line.strip()
                if stripped.startswith("suite_name:"):
                    return stripped.split(":", 1)[1].strip().strip("'\"")
    fallback = {
        "smoke": "mnist_mlp3_tangent_rg_v1_smoke",
        "pilot_1000_epochs": "mnist_mlp3_tangent_rg_v1_pilot1000",
        "long_horizon_10000_epochs": "mnist_mlp3_tangent_rg_v1_reference10000",
    }
    if PROFILE not in fallback:
        raise FileNotFoundError(
            f"Cannot derive suite_name for PROFILE={PROFILE!r}; set CONFIG_PATH "
            "or PROTOCOL_SLUG explicitly."
        )
    return fallback[PROFILE]

PROTOCOL_SLUG = _suite_name_from_profile()
OUTPUT_ROOT_PATH = Path(
    OUTPUT_ROOT or RUN_ROOT_PATH / PROTOCOL_SLUG / "notebook_outputs"
).expanduser().resolve()
OUTPUT_ROOT_PATH.mkdir(parents=True, exist_ok=True)

SEEDS = tuple(int(seed) for seed in SEEDS)
if SEEDS != (1337, 2027, 31415):
    print("WARNING: this is not the preregistered three-seed tuple:", SEEDS)

print("repository:", REPO_ROOT)
print("run root:", RUN_ROOT_PATH)
print("tail checkpoint cache root:", CHECKPOINT_CACHE_ROOT_PATH)
print("effective suite:", PROTOCOL_SLUG)
print("output root:", OUTPUT_ROOT_PATH)
print("seeds:", SEEDS)


In [ ]:
from rg_baselines.statistics import summarize_numeric_metrics
from rg_baselines.tangent_rg import powerlaw_fit, trace_log


from rg_baselines.tangent_rg import (
    AdamWProfile,
    MuonClipRMSProfile,
    MuonProfile,
    TangentRGConfig,
    load_analysis_checkpoint,
    list_analysis_checkpoints,
    list_capture_files,
    load_step_capture,
    replay_calibrated_step,
)
from rg_baselines.tangent_rg.checkpoints import load_verified_tail_checkpoint_refs
from rg_baselines.tangent_rg.protocol import tail_checkpoint_epochs
from rg_baselines.tangent_rg import nulls, polar, single_checkpoint, stiefel, two_checkpoint

from rg_baselines.tangent_rg import ecs_jacobians


## Five additional weight-only Jacobians

**`operator_kind`: `additional_single_checkpoint_ecs_jacobian_bundle`**

**`map_definition`: `Exact derivatives of five explicitly defined spectral maps; all headline spectra are eigenvalues of J*J.`**

**Identifiability caveat.** These maps are candidate RG observables, not an inferred training-flow Jacobian. The Feshbach map is first-order shell-blind in the checkpoint SVD gauge because B=0.

These strings are persisted with every result row. A visually useful
spectrum does not change the identity of the map that produced it.


In [ ]:
def require_path(path, *, description="artifact"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}\n"
            "Run the prerequisite numbered notebook or set RUN_ROOT / "
            "OUTPUT_ROOT to the completed protocol directory."
        )
    return path


def first_existing(directory, names, *, description):
    directory = Path(directory)
    candidates = [directory / name for name in names]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {description} beneath {directory}. Expected one of:\n"
        + "\n".join(f"  - {path}" for path in candidates)
    )


def resolve_protocol_root():
    direct = RUN_ROOT_PATH / PROTOCOL_SLUG
    return direct if direct.is_dir() else RUN_ROOT_PATH


def resolve_arm_dir(optimizer_slug):
    protocol = resolve_protocol_root()
    candidates = [
        protocol / optimizer_slug,
        protocol / "results" / optimizer_slug,
        RUN_ROOT_PATH / optimizer_slug,
        RUN_ROOT_PATH / "results" / optimizer_slug,
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    if REQUIRE_ARTIFACTS:
        raise FileNotFoundError(
            f"No completed {optimizer_slug!r} arm was found. Checked:\n"
            + "\n".join(f"  - {path}" for path in candidates)
        )
    return candidates[0]


def resolve_seed_dir(optimizer_slug, seed):
    arm = resolve_arm_dir(optimizer_slug)
    candidates = [
        arm / f"seed_{int(seed)}",
        arm / f"seed_{int(seed):05d}",
        arm / "seeds" / f"seed_{int(seed)}",
        arm / "seeds" / f"seed_{int(seed):05d}",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Missing seed directory for optimizer={optimizer_slug}, seed={seed}. "
        f"Checked {candidates}."
    )


def validate_run_identity(seed_dir, *, optimizer_slug, seed):
    seed_dir = Path(seed_dir)
    manifest = json.loads(
        require_path(seed_dir / "manifest.json", description="run manifest")
        .read_text(encoding="utf-8")
    )
    resolved = json.loads(
        require_path(seed_dir / "resolved_config.json", description="resolved config")
        .read_text(encoding="utf-8")
    )
    completion = json.loads(
        require_path(seed_dir / "run_complete.json", description="completion marker")
        .read_text(encoding="utf-8")
    )
    config = dict(resolved.get("config", resolved))
    checks = {
        "manifest suite": (manifest.get("suite_name"), PROTOCOL_SLUG),
        "resolved suite": (config.get("suite_name"), PROTOCOL_SLUG),
        "manifest optimizer": (manifest.get("optimizer"), optimizer_slug),
        "resolved optimizer": (config.get("optimizer"), optimizer_slug),
        "completion optimizer": (completion.get("optimizer"), optimizer_slug),
        "manifest seed": (manifest.get("seed"), int(seed)),
        "resolved seed": (config.get("seed"), int(seed)),
        "completion seed": (completion.get("seed"), int(seed)),
    }
    mismatches = [
        f"{label}: observed={observed!r}, expected={expected!r}"
        for label, (observed, expected) in checks.items()
        if str(observed) != str(expected)
    ]
    fingerprints = {
        str(manifest.get("protocol_fingerprint", "")),
        str(resolved.get("protocol_fingerprint", "")),
        str(completion.get("protocol_fingerprint", "")),
    }
    if "" in fingerprints or len(fingerprints) != 1:
        mismatches.append(
            "manifest/resolved/completion protocol fingerprints are missing or unequal"
        )
    if not bool(completion.get("completed", False)):
        mismatches.append("run_complete.json does not declare completed=true")
    try:
        resolved_epochs = int(config["epochs"])
        completion_epochs = int(completion["epochs"])
        completion_step = int(completion["global_step"])
        best_validation_epoch = int(completion["best_validation_epoch"])
        analysis_plan = dict(resolved["analysis_plan"])
        plan_steps_per_epoch = int(analysis_plan["steps_per_epoch"])
        plan_total_steps = int(analysis_plan["total_steps"])
    except (KeyError, TypeError, ValueError) as error:
        mismatches.append(
            "resolved/completion final-horizon metadata is missing or invalid: "
            f"{type(error).__name__}: {error}"
        )
    else:
        if resolved_epochs < 1 or plan_steps_per_epoch < 1:
            mismatches.append("resolved epochs and steps_per_epoch must be positive")
        if plan_total_steps != resolved_epochs * plan_steps_per_epoch:
            mismatches.append(
                "resolved analysis_plan total_steps does not equal "
                "epochs * steps_per_epoch"
            )
        if completion_epochs != resolved_epochs:
            mismatches.append(
                f"completion epochs={completion_epochs} != resolved epochs={resolved_epochs}"
            )
        if completion_step != plan_total_steps:
            mismatches.append(
                f"completion global_step={completion_step} != resolved "
                f"analysis_plan total_steps={plan_total_steps}"
            )
        if not 0 <= best_validation_epoch <= resolved_epochs:
            mismatches.append(
                f"best_validation_epoch={best_validation_epoch} is outside "
                f"[0, {resolved_epochs}]"
            )
    if mismatches:
        raise RuntimeError(
            f"Run identity/provenance mismatch beneath {seed_dir}:\n  - "
            + "\n  - ".join(mismatches)
        )
    return manifest, resolved, completion


def validate_cross_run_provenance(manifests):
    manifests = list(manifests)
    if not manifests:
        raise RuntimeError("No manifests supplied for cross-run provenance audit")
    invariant_fields = (
        "suite_name", "dataset", "model", "initialization", "normalization",
        "train_indices_sha256", "validation_indices_sha256",
        "test_monitoring_only", "analysis_plan", "device", "software_versions",
        "determinism_settings",
    )
    disagreements = []
    for field in invariant_fields:
        serialized = {
            json.dumps(item.get(field), sort_keys=True, default=str)
            for item in manifests
        }
        if len(serialized) != 1:
            disagreements.append(field)
    if disagreements:
        raise RuntimeError(
            "Matched arms disagree on frozen run provenance fields: "
            + ", ".join(disagreements)
        )
    identities = {
        (str(item.get("optimizer")), int(item.get("seed"))) for item in manifests
    }
    expected = {
        (str(optimizer), int(seed))
        for optimizer in OPTIMIZER_SLUGS
        for seed in SEEDS
    } if "OPTIMIZER_SLUGS" in globals() else identities
    if identities != expected:
        raise RuntimeError(
            f"Manifest optimizer/seed grid is incomplete: observed={sorted(identities)}, "
            f"expected={sorted(expected)}"
        )
    return pd.DataFrame([
        {
            "optimizer": item.get("optimizer"),
            "seed": item.get("seed"),
            "device": item.get("device"),
            "software_versions": json.dumps(
                item.get("software_versions"), sort_keys=True, default=str
            ),
            "determinism_settings": json.dumps(
                item.get("determinism_settings"), sort_keys=True, default=str
            ),
            "pooling_compatibility_policy": (
                "headline pooling requires identical device, software versions, "
                "determinism settings, and scientific invariants across all runs"
            ),
        }
        for item in manifests
    ])


def record_dict(value):
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, dict):
        return dict(value)
    if hasattr(value, "__dict__"):
        return dict(vars(value))
    raise TypeError(f"Cannot convert {type(value).__name__} to an audit row")


def records_from_result(result):
    if result is None:
        return []
    if is_dataclass(result):
        return [record_dict(result)]
    if isinstance(result, dict):
        if "operator_kind" in result:
            return [dict(result)]
        rows = []
        for value in result.values():
            rows.extend(records_from_result(value))
        return rows
    if isinstance(result, (tuple, list)):
        rows = []
        for value in result:
            rows.extend(records_from_result(value))
        return rows
    return [record_dict(result)]


def spectrum_from_record(row):
    for name in (
        "spectrum", "eigenvalues", "singular_values", "rates",
        "positive_spectrum", "gram_spectrum",
    ):
        if name in row:
            values = np.asarray(row[name], dtype=float).reshape(-1)
            return values[np.isfinite(values) & (values > 0.0)]
    raise KeyError(
        "Operator record contains no recognized positive spectrum field. "
        f"Available fields: {sorted(row)}"
    )


def positive_spectrum(values, *, minimum_count=2):
    sample = np.asarray(values, dtype=float).reshape(-1)
    sample = sample[np.isfinite(sample) & (sample > 0.0)]
    sample = np.sort(sample)
    if sample.size < int(minimum_count):
        raise ValueError(
            f"Need at least {minimum_count} finite positive spectral values; "
            f"found {sample.size}."
        )
    return sample


def fit_spectrum_with_trace(
    values,
    *,
    operator_kind,
    map_definition,
    spectrum_kind,
    metadata,
    top_k_values=(0, 1, 2, 3, 4, 5),
    minimum_tail=8,
):
    # Fit amplitudes once, transform that fit to energy, and audit trace-log.
    # The power-law package is never called independently on squared values.
    # Trace-log uses squared values at the amplitude fit's independent rank.

    if str(spectrum_kind) != "amplitude":
        raise ValueError(
            "fit_spectrum_with_trace accepts operator amplitudes only; "
            "energy rows are produced by the exact amplitude-to-energy transform."
        )

    sample = positive_spectrum(values)
    feasible_top_k = tuple(
        int(value) for value in top_k_values if int(value) <= sample.size - 2
    )
    if not feasible_top_k or feasible_top_k[0] != 0:
        feasible_top_k = (0,)
    amplitude_fits = powerlaw_fit.fit_clipping_sensitivity(
        sample,
        top_k_values=feasible_top_k,
        minimum_tail=int(minimum_tail),
        operator_kind=str(operator_kind),
        map_definition=str(map_definition),
        spectrum_kind="amplitude",
        metadata=dict(metadata),
    )
    energy_rows = [
        powerlaw_fit.amplitude_fit_to_energy(row)
        for row in amplitude_fits.to_dict(orient="records")
    ]
    fits = pd.concat(
        [amplitude_fits, pd.DataFrame(energy_rows)],
        ignore_index=True,
        sort=False,
    )
    primary = amplitude_fits.loc[amplitude_fits["clip_top_k"].eq(0)].iloc[0]
    energy = sample ** 2
    trace_row = {
        **dict(metadata),
        "operator_kind": str(operator_kind),
        "map_definition": str(map_definition),
        "spectrum_kind": "energy_derived_from_amplitude",
        "support_rank_source": "powerlaw.Fit package-selected xmin tail count",
        "support_selected_from_same_trace_log": False,
        "support_rank": int(primary.get("n_tail", 0)),
        "trace_log_total": np.nan,
        "trace_log_per_eval": np.nan,
        "lambda_cut_scaled": np.nan,
        "trace_status": "fit_has_no_supported_tail",
    }
    rank = int(primary.get("n_tail", 0))
    if rank > 0:
        evaluated = trace_log.trace_log_at_rank(
            energy,
            rank=min(rank, energy.size),
            normalization_dimension=float(energy.size),
            rank_source="powerlaw.Fit package-selected xmin tail count",
        )
        trace_row.update(evaluated)
        trace_row["trace_status"] = "ok"
    return fits, pd.DataFrame([trace_row])


def save_analysis_frames(method_slug, *, operators, fits, traces):
    destination = OUTPUT_ROOT_PATH / "analyses" / str(method_slug)
    destination.mkdir(parents=True, exist_ok=True)
    required_identity = {
        "optimizer", "seed", "protocol_fingerprint", "source_artifact_kind"
    }
    for label, frame in (("operators", operators), ("fits", fits), ("traces", traces)):
        missing = required_identity - set(frame.columns)
        if missing:
            raise RuntimeError(
                f"{method_slug} {label} lack analysis provenance: {sorted(missing)}"
            )
        if frame[list(required_identity)].isna().any().any():
            raise RuntimeError(f"{method_slug} {label} contain null analysis provenance")
    identity_rows = fits[
        ["optimizer", "seed", "protocol_fingerprint", "source_artifact_kind"]
    ].drop_duplicates()
    duplicate_fingerprints = (
        identity_rows.groupby(["optimizer", "seed"], dropna=False)[
            "protocol_fingerprint"
        ].nunique()
    )
    if (duplicate_fingerprints != 1).any():
        raise RuntimeError(
            f"{method_slug} has multiple protocol fingerprints for one optimizer/seed"
        )
    fingerprint_grid = {
        f"{row.optimizer}:{int(row.seed)}": str(row.protocol_fingerprint)
        for row in identity_rows.itertuples(index=False)
    }
    expected_grid_count = identity_rows[["optimizer", "seed"]].drop_duplicates().shape[0]
    if len(fingerprint_grid) != expected_grid_count:
        raise RuntimeError(f"{method_slug} fingerprint-grid keys are not unique")
    provenance_manifest = {
        "schema_version": 1,
        "suite_name": str(PROTOCOL_SLUG),
        "method_slug": str(method_slug),
        "optimizer_seed_protocol_fingerprints": dict(sorted(fingerprint_grid.items())),
        "source_artifact_kinds": sorted(
            identity_rows["source_artifact_kind"].astype(str).unique().tolist()
        ),
        "operator_row_count": int(len(operators)),
        "fit_row_count": int(len(fits)),
        "trace_row_count": int(len(traces)),
    }
    provenance_manifest["analysis_contract_tokens"] = sorted(
        fits["analysis_contract_token"].dropna().astype(str).unique().tolist()
        if "analysis_contract_token" in fits.columns
        else []
    )
    operators.to_csv(destination / "operator_rows.csv", index=False)
    fits.to_csv(destination / "powerlaw_fits.csv", index=False)
    traces.to_csv(destination / "trace_log_independent_support.csv", index=False)
    (destination / "method_provenance.json").write_text(
        json.dumps(provenance_manifest, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    return destination


def save_spectrum_ccdf_gallery(
    spectral_arrays,
    *,
    method_slug,
    maximum_panels=24,
):
    # Save bounded log-log PDF/CCDF diagnostics for positive amplitudes.
    gallery = OUTPUT_ROOT_PATH / "analyses" / str(method_slug) / "spectrum_pdf_ccdf"
    gallery.mkdir(parents=True, exist_ok=True)
    rows = []
    for index, (key, raw) in enumerate(sorted(spectral_arrays.items())):
        if index >= int(maximum_panels):
            break
        sample = positive_spectrum(raw)
        x, ccdf = powerlaw_fit.empirical_ccdf(sample)
        fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.0))
        if sample[0] < sample[-1]:
            bins = np.geomspace(sample[0], sample[-1], min(50, max(8, sample.size // 3)))
            axes[0].hist(sample, bins=bins, density=True, histtype="step", linewidth=1.8)
        else:
            axes[0].scatter(sample, np.ones_like(sample), s=15)
        axes[1].step(x, ccdf, where="post", linewidth=1.8)
        for axis in axes:
            axis.set_xscale("log")
            axis.set_yscale("log")
            axis.grid(alpha=0.2)
        axes[0].set(xlabel="amplitude b", ylabel="density", title="PDF")
        axes[1].set(xlabel="amplitude b", ylabel="P(B >= b)", title="CCDF")
        fig.suptitle(str(key), fontsize=8)
        fig.tight_layout()
        safe = "".join(character if character.isalnum() or character in "-_" else "_" for character in str(key))
        path = gallery / f"{index:03d}_{safe[:160]}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        if SHOW_PLOTS and index < 3:
            plt.show()
        else:
            plt.close(fig)
        rows.append({"spectrum_key": str(key), "n_positive": int(sample.size), "figure": str(path)})
    index_frame = pd.DataFrame(rows)
    index_frame.to_csv(gallery / "index.csv", index=False)
    return index_frame


def plot_fit_alpha_ci(fits, *, method_slug, title):
    usable = fits.copy()
    if "fit_ok" in usable:
        usable = usable[boolean_series(usable["fit_ok"])]
    if "spectrum_kind" in usable:
        energy = usable[
            usable["spectrum_kind"].astype(str).eq("energy_derived_from_amplitude")
        ]
        if not energy.empty:
            usable = energy
    primary = usable[usable["clip_top_k"].eq(0)] if "clip_top_k" in usable else usable
    if primary.empty:
        raise RuntimeError(
            f"{method_slug}: no successful preregistered raw fits; inspect powerlaw_fits.csv"
        )
    if "state_index" not in primary:
        primary["state_index"] = 0
    groups = tuple(
        name
        for name in (
            "optimizer", "layer", "method", "null_kind", "pair_stride",
            "epsilon", "evidence_role",
        )
        if name in primary
    )
    if not groups:
        primary["method"] = str(method_slug)
        groups = ("method",)
    return plot_seed_ci(
        primary,
        x="state_index",
        metric="alpha",
        groups=groups,
        title=title,
        ylabel="Power-law density exponent alpha",
        reference=2.0,
        allow_incomplete=True,
        incomplete_output_path=(
            OUTPUT_ROOT_PATH / "analyses" / str(method_slug)
            / "incomplete_alpha_ci_groups.csv"
        ),
        output_path=(
            OUTPUT_ROOT_PATH / "analyses" / str(method_slug) / "alpha_95ci.png"
        ),
    )


def call_supported(function, /, *args, **kwargs):
    signature = inspect.signature(function)
    if any(
        parameter.kind is inspect.Parameter.VAR_KEYWORD
        for parameter in signature.parameters.values()
    ):
        return function(*args, **kwargs)
    supported = {key: value for key, value in kwargs.items() if key in signature.parameters}
    return function(*args, **supported)


def boolean_series(values):
    if getattr(values, "dtype", None) == bool:
        return values
    return values.astype(str).str.strip().str.lower().isin({"1", "true", "yes"})


def ci_summary(
    frame,
    *,
    groups,
    metrics,
    allow_incomplete=False,
    incomplete_output_path=None,
    return_incomplete=False,
):
    missing = set((*groups, *metrics, "seed")) - set(frame.columns)
    if missing:
        raise ValueError(f"CI input is missing columns: {sorted(missing)}")
    # Repeated layers/checkpoints/probes are not independent replicates.  First
    # collapse every declared group to one value per complete training seed.
    replicate = (
        frame.groupby([*groups, "seed"], as_index=False, dropna=False)[list(metrics)]
        .mean(numeric_only=True)
    )
    summary = summarize_numeric_metrics(
        replicate,
        group_columns=tuple(groups),
        metrics=tuple(metrics),
        confidence=0.95,
    )
    if summary.empty:
        raise RuntimeError("Confidence-interval summary is empty after seed aggregation")
    incomplete = summary[pd.to_numeric(summary["n"], errors="coerce") != len(SEEDS)]
    if not incomplete.empty:
        if incomplete_output_path is not None:
            incomplete_output_path = Path(incomplete_output_path)
            incomplete_output_path.parent.mkdir(parents=True, exist_ok=True)
            incomplete.to_csv(incomplete_output_path, index=False)
        if allow_incomplete:
            print(
                "WARNING: dropping incomplete CI identities from the mean/band; "
                "faint individual-seed traces remain visible.\n"
                + incomplete[
                    [name for name in (*groups, "metric", "n") if name in incomplete]
                ].to_string(index=False)
            )
        else:
            identity = [name for name in (*groups, "metric", "n") if name in incomplete]
            raise RuntimeError(
                "Every confidence-interval row requires exactly the preregistered "
                f"{len(SEEDS)} complete seeds. Incomplete identities:\n"
                + incomplete[identity].to_string(index=False)
            )
    complete = summary[pd.to_numeric(summary["n"], errors="coerce") == len(SEEDS)].copy()
    if return_incomplete:
        return complete, incomplete.copy()
    return complete


def plot_seed_ci(
    frame,
    *,
    x,
    metric,
    groups,
    title,
    ylabel,
    reference=None,
    output_path=None,
    allow_incomplete=False,
    incomplete_output_path=None,
):
    groups = tuple(groups)
    if allow_incomplete and incomplete_output_path is None and output_path is not None:
        output_path_for_report = Path(output_path)
        incomplete_output_path = output_path_for_report.with_name(
            output_path_for_report.stem + "_incomplete_ci_groups.csv"
        )
    summary, incomplete = ci_summary(
        frame,
        groups=(*groups, x),
        metrics=(metric,),
        allow_incomplete=allow_incomplete,
        incomplete_output_path=incomplete_output_path,
        return_incomplete=True,
    )
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    if not groups:
        frame = frame.copy()
        frame["series"] = "all"
        groups = ("series",)
    for identity, group in frame.groupby(list(groups), dropna=False):
        identity = identity if isinstance(identity, tuple) else (identity,)
        label = ", ".join(f"{key}={value}" for key, value in zip(groups, identity))
        for _, seed_frame in group.groupby("seed"):
            ordered = (
                seed_frame.groupby(x, as_index=False, dropna=False)[metric]
                .mean(numeric_only=True)
                .sort_values(x)
            )
            ax.plot(ordered[x], ordered[metric], alpha=0.16, linewidth=0.9)
        selected = summary.copy()
        for key, value in zip(groups, identity):
            selected = selected[selected[key].astype(str) == str(value)]
        selected = selected[selected["metric"] == metric].sort_values(x)
        if selected.empty:
            pass
        else:
            xv = selected[x].to_numpy(dtype=float)
            mean = selected["mean"].to_numpy(dtype=float)
            low = selected["ci_low"].to_numpy(dtype=float)
            high = selected["ci_high"].to_numpy(dtype=float)
            ax.plot(xv, mean, marker="o", linewidth=2.1, label=label)
            finite = np.isfinite(low) & np.isfinite(high)
            ax.fill_between(xv[finite], low[finite], high[finite], alpha=0.18)
        missing = incomplete.copy()
        for key, value in zip(groups, identity):
            missing = missing[missing[key].astype(str) == str(value)]
        missing = missing[missing["metric"] == metric]
        if not missing.empty:
            ax.scatter(
                missing[x].to_numpy(dtype=float),
                missing["mean"].to_numpy(dtype=float),
                marker="x", color="#555555", alpha=0.65, zorder=4,
            )
    if reference is not None:
        ax.axhline(float(reference), color="#333333", linestyle="--", linewidth=1.4)
    ax.set(xlabel=x, ylabel=ylabel, title=title)
    ax.set_xscale("symlog", linthresh=1.0)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    return summary, fig


_VERIFIED_TAIL_CACHE_REFS = {}
_VERIFIED_TAIL_CHECKPOINT_IDENTITIES = {}
_VERIFIED_RUN_IDENTITIES = {}


def require_complete_seed(optimizer_slug, seed):
    seed_dir = resolve_seed_dir(optimizer_slug, seed)
    manifest, _, _ = validate_run_identity(
        seed_dir, optimizer_slug=optimizer_slug, seed=seed
    )
    _VERIFIED_RUN_IDENTITIES[(str(optimizer_slug), int(seed))] = {
        "protocol_fingerprint": str(manifest["protocol_fingerprint"]),
        "source_seed_dir": str(Path(seed_dir).resolve()),
    }
    return seed_dir


def verified_run_fingerprint(optimizer_slug, seed):
    identity = _VERIFIED_RUN_IDENTITIES.get((str(optimizer_slug), int(seed)))
    if identity is None:
        raise RuntimeError(
            f"Run identity was not verified for optimizer={optimizer_slug}, seed={seed}"
        )
    return str(identity["protocol_fingerprint"])


def require_tail_checkpoint_cache(optimizer_slug, seed):
    # Establish expected identity from the separately completed run. Never
    # trust identity claimed only by the temporary cache itself.
    source_seed_dir = resolve_seed_dir(optimizer_slug, seed)
    manifest, resolved, completion = validate_run_identity(
        source_seed_dir, optimizer_slug=optimizer_slug, seed=seed
    )
    _VERIFIED_RUN_IDENTITIES[(str(optimizer_slug), int(seed))] = {
        "protocol_fingerprint": str(manifest["protocol_fingerprint"]),
        "source_seed_dir": str(Path(source_seed_dir).resolve()),
    }
    resolved_values = dict(resolved.get("config", resolved))
    if "epochs" not in resolved_values:
        raise KeyError(f"Resolved run config lacks epochs: {source_seed_dir}")
    expected_epochs = tail_checkpoint_epochs(int(resolved_values["epochs"]))
    recorded_cache_root = Path(
        resolved_values.get(
            "tail_checkpoint_cache_root",
            "/tmp/rg-mnist-mlp3-tangent-checkpoints",
        )
    ).expanduser().resolve()
    if recorded_cache_root != CHECKPOINT_CACHE_ROOT_PATH:
        raise RuntimeError(
            "Notebook CHECKPOINT_CACHE_ROOT disagrees with the completed run: "
            f"notebook={CHECKPOINT_CACHE_ROOT_PATH}, recorded={recorded_cache_root}. "
            "Set CHECKPOINT_CACHE_ROOT (or "
            "RG_MNIST_TANGENT_CHECKPOINT_CACHE_ROOT) to the recorded cache root."
        )
    temporary_root = Path("/tmp").resolve()
    if (
        recorded_cache_root == temporary_root
        or not recorded_cache_root.is_relative_to(temporary_root)
    ):
        raise RuntimeError(
            f"Tail checkpoint cache must be a safe child of /tmp: {recorded_cache_root}"
        )
    cache_seed_dir = (
        recorded_cache_root
        / PROTOCOL_SLUG
        / str(optimizer_slug)
        / f"seed_{int(seed)}"
    ).resolve()
    refs = tuple(
        load_verified_tail_checkpoint_refs(
            cache_seed_dir,
            expected_suite_name=PROTOCOL_SLUG,
            expected_optimizer_name=str(optimizer_slug),
            expected_seed=int(seed),
            expected_fingerprint=str(manifest["protocol_fingerprint"]),
            expected_epochs=expected_epochs,
            validate_payloads=True,
        )
    )
    expected_count = min(100, int(resolved_values["epochs"]))
    if len(refs) != expected_count:
        raise RuntimeError(
            f"Verified cache count {len(refs)} != expected {expected_count}: "
            f"{cache_seed_dir}"
        )
    completed_epochs = int(completion.get("epochs", -1))
    completed_step = int(completion.get("global_step", -1))
    if completed_epochs != int(resolved_values["epochs"]) or completed_step < 1:
        raise RuntimeError(
            f"Completed run horizon is inconsistent beneath {source_seed_dir}"
        )
    steps_per_epoch, remainder = divmod(completed_step, completed_epochs)
    if remainder or steps_per_epoch < 1:
        raise RuntimeError(
            f"Completed step count is not an exact epoch grid beneath {source_seed_dir}"
        )
    expected_pairs = tuple(
        (int(epoch), int(epoch) * int(steps_per_epoch))
        for epoch in expected_epochs
    )
    observed_pairs = tuple((int(ref.epoch), int(ref.global_step)) for ref in refs)
    completion_cache_dir = completion.get("tail_checkpoint_cache_dir")
    if not isinstance(completion_cache_dir, str) or not completion_cache_dir.strip():
        raise RuntimeError(
            f"Completion marker lacks tail_checkpoint_cache_dir: {source_seed_dir}"
        )
    source_cache_checks = {
        "cache_dir": (
            str(Path(completion_cache_dir).expanduser().resolve()),
            str(cache_seed_dir),
        ),
        "checkpoint_count": (completion.get("tail_checkpoint_count"), expected_count),
        "first_epoch": (completion.get("tail_checkpoint_first_epoch"), expected_epochs[0]),
        "last_epoch": (completion.get("tail_checkpoint_last_epoch"), expected_epochs[-1]),
        "epoch_step_grid": (observed_pairs, expected_pairs),
    }
    mismatches = [
        f"{label}: observed={observed!r}, expected={expected!r}"
        for label, (observed, expected) in source_cache_checks.items()
        if observed != expected
    ]
    if mismatches:
        raise RuntimeError(
            "Tail cache disagrees with the persistent run completion marker:\n  - "
            + "\n  - ".join(mismatches)
        )
    resolved_cache_dir = cache_seed_dir.resolve()
    _VERIFIED_TAIL_CACHE_REFS[resolved_cache_dir] = refs
    for ref in refs:
        _VERIFIED_TAIL_CHECKPOINT_IDENTITIES[ref.path.resolve()] = {
            "protocol_fingerprint": str(manifest["protocol_fingerprint"]),
            "optimizer": str(optimizer_slug),
            "seed": int(seed),
            "epoch": int(ref.epoch),
            "global_step": int(ref.global_step),
            "cache_seed_dir": str(resolved_cache_dir),
        }
    return resolved_cache_dir




def analysis_checkpoint_refs(seed_dir):
    cache_seed_dir = Path(seed_dir).resolve()
    refs = tuple(_VERIFIED_TAIL_CACHE_REFS.get(cache_seed_dir, ()))
    if len(refs) < 2:
        raise RuntimeError(
            "Tail checkpoint cache was not verified in this kernel, or contains "
            f"fewer than two states: {cache_seed_dir}. Call "
            "require_tail_checkpoint_cache(optimizer, seed); analysis notebooks "
            "never fall back to sparse run checkpoints or training."
        )
    return refs


def selected_checkpoint_pairs(seed_dir, *, maximum_pairs=None, stride=1):
    refs = analysis_checkpoint_refs(seed_dir)
    stride = int(stride)
    if stride < 1:
        raise ValueError("PAIR_STRIDE must be positive")
    all_pairs = list(zip(refs[:-stride], refs[stride:]))
    if not all_pairs:
        raise RuntimeError(f"No checkpoint pairs selected from {seed_dir}")
    budget = (
        len(all_pairs)
        if maximum_pairs is None
        else min(int(maximum_pairs), len(all_pairs))
    )
    if budget < 1:
        raise ValueError("maximum_pairs must be positive or None")
    if budget == len(all_pairs):
        indices = list(range(len(all_pairs)))
        complete_role = (
            "complete_verified_tail_adjacent_grid"
            if stride == 1
            else "complete_verified_tail_stride_grid"
        )
        rule = (
            "all chronological pairs from the verified tail checkpoint cache: "
            f"stride={stride}, selected_count={len(indices)}, "
            f"total_available={len(all_pairs)}"
        )
        roles = [complete_role for _ in indices]
    else:
        tail_count = min(max(2, budget // 3), budget)
        broad_budget = max(0, budget - tail_count)
        broad_limit = max(0, len(all_pairs) - tail_count)
        broad_indices = []
        if broad_budget and broad_limit:
            broad_indices = np.unique(
                np.rint(np.geomspace(1, broad_limit, num=broad_budget) - 1).astype(int)
            ).tolist()
            if 0 not in broad_indices:
                broad_indices.insert(0, 0)
        tail_indices = list(range(len(all_pairs) - tail_count, len(all_pairs)))
        indices = sorted(set(broad_indices + tail_indices))
        rule = (
            "deterministic log-index broad pair sample plus consecutive pair tail: "
            f"stride={stride}, requested={maximum_pairs}, selected_indices={indices}, "
            f"tail_count={tail_count}, total_available={len(all_pairs)}"
        )
        tail_start = len(all_pairs) - tail_count
        roles = [
            (
                "consecutive_tail"
                if stride == 1 and index >= tail_start
                else "stride_tail"
                if index >= tail_start
                else "log_index_broad"
            )
            for index in indices
        ]
    return [
        (
            all_pairs[index][0], all_pairs[index][1], rule, role,
        )
        for index, role in zip(indices, roles)
    ]


def selected_checkpoint_pairs_for_strides(seed_dir, *, strides, maximum_pairs=None):
    selected = []
    available_states = len(analysis_checkpoint_refs(seed_dir))
    for stride in tuple(dict.fromkeys(int(value) for value in strides)):
        if stride >= available_states:
            print(
                f"Skipping pair stride={stride}: verified cache has only "
                f"{available_states} states"
            )
            continue
        for previous, current, rule, role in selected_checkpoint_pairs(
            seed_dir, maximum_pairs=maximum_pairs, stride=stride
        ):
            selected.append((stride, previous, current, rule, role))
    if not selected:
        raise RuntimeError(f"No multi-spacing checkpoint pairs selected from {seed_dir}")
    return selected


def checkpoint_refs_at_epoch_stride(seed_dir, *, epoch_stride=1):
    refs = analysis_checkpoint_refs(seed_dir)
    stride = int(epoch_stride)
    if stride < 1:
        raise ValueError("ANALYSIS_EPOCH_STRIDE must be positive")
    if stride == 1:
        return refs
    selected = tuple(
        ref for ref in refs
        if int(ref.epoch) > 0 and int(ref.epoch) % stride == 0
    )
    if len(selected) < 2:
        raise RuntimeError(
            f"Epoch stride {stride} selected fewer than two checkpoints from "
            f"{seed_dir}: epochs={[int(ref.epoch) for ref in selected]}"
        )
    return selected


def selected_trajectory_matrices(
    seed_dir, *, layers, maximum_checkpoints, epoch_stride=1
):
    refs = checkpoint_refs_at_epoch_stride(
        seed_dir, epoch_stride=epoch_stride
    )
    budget = min(int(maximum_checkpoints), len(refs))
    if budget == len(refs):
        indices = list(range(len(refs)))
        roles = ["complete_verified_tail_state_grid" for _ in indices]
        rule = (
            "all chronological states from the verified tail checkpoint cache "
            "after exact positive epoch-modulus filtering: "
            f"epoch_stride={int(epoch_stride)}, selected_count={len(indices)}, "
            f"total_stride_eligible={len(refs)}"
        )
    else:
        tail_count = min(max(2, budget // 3), budget)
        broad_budget = max(0, budget - tail_count)
        broad_limit = max(0, len(refs) - tail_count)
        broad_indices = []
        if broad_budget and broad_limit:
            broad_indices = np.unique(
                np.rint(np.geomspace(1, broad_limit, num=broad_budget) - 1)
                .astype(int)
            ).tolist()
            if 0 not in broad_indices:
                broad_indices.insert(0, 0)
        tail_indices = list(range(len(refs) - tail_count, len(refs)))
        indices = sorted(set(broad_indices + tail_indices))
        tail_start = len(refs) - tail_count
        roles = [
            "consecutive_tail" if index >= tail_start else "log_index_broad"
            for index in indices
        ]
        rule = (
            f"deterministic log-index broad sample plus consecutive tail: "
            f"requested={maximum_checkpoints}, selected_indices={indices}, "
            f"tail_count={tail_count}, total_available={len(refs)}"
        )
    selected = [refs[index] for index in indices]
    if len(selected) < 2:
        raise RuntimeError(f"Need at least two trajectory states beneath {seed_dir}")
    for index, ref, role in zip(indices, selected, roles):
        for layer in layers:
            yield ref, str(layer), checkpoint_matrix(ref.path, str(layer)), rule, role


@lru_cache(maxsize=int(CHECKPOINT_PAYLOAD_CACHE_SIZE))
def _load_verified_checkpoint_payload_cached(
    source_text,
    expected_fingerprint,
    expected_optimizer,
    expected_seed,
    expected_epoch,
    expected_global_step,
):
    source = Path(source_text)
    payload = load_analysis_checkpoint(
        source, expected_fingerprint=str(expected_fingerprint)
    )
    checks = {
        "optimizer": (payload.get("optimizer"), str(expected_optimizer)),
        "seed": (payload.get("seed"), int(expected_seed)),
        "epoch": (payload.get("epoch"), int(expected_epoch)),
        "global_step": (payload.get("global_step"), int(expected_global_step)),
    }
    mismatches = [
        f"{field}: observed={observed!r}, expected={expected!r}"
        for field, (observed, expected) in checks.items()
        if str(observed) != str(expected)
    ]
    if mismatches:
        raise RuntimeError(
            f"Checkpoint payload disagrees with verified cache identity: {source}; "
            + "; ".join(mismatches)
        )
    return payload


def load_checkpoint_payload(path):
    source = require_path(path, description="verified tail checkpoint").resolve()
    expected = _VERIFIED_TAIL_CHECKPOINT_IDENTITIES.get(source)
    if expected is None:
        raise RuntimeError(
            f"Checkpoint was not admitted by the strict tail-cache verifier: {source}. "
            "Analysis never falls back to an arbitrary checkpoint path."
        )
    payload = _load_verified_checkpoint_payload_cached(
        str(source),
        str(expected["protocol_fingerprint"]),
        str(expected["optimizer"]),
        int(expected["seed"]),
        int(expected["epoch"]),
        int(expected["global_step"]),
    )
    match = re.fullmatch(
        r"analysis_epoch_(?P<epoch>\d+)_step_(?P<step>\d+)\.pt", source.name
    )
    if match is None:
        raise RuntimeError(f"Expected an immutable analysis-checkpoint filename: {source}")
    if (
        int(payload.get("epoch", -1)) != int(match.group("epoch"))
        or int(payload.get("global_step", -1)) != int(match.group("step"))
        or int(payload.get("epoch", -1)) != int(expected["epoch"])
        or int(payload.get("global_step", -1)) != int(expected["global_step"])
    ):
        raise RuntimeError(f"Checkpoint payload epoch/step disagrees with filename: {source}")
    return payload


def checkpoint_state_dict(payload):
    for key in ("model", "model_state_dict", "state_dict"):
        value = payload.get(key) if isinstance(payload, dict) else None
        if isinstance(value, dict):
            return value
    raise KeyError(
        "Checkpoint has no model/model_state_dict/state_dict mapping; "
        f"available keys={list(payload) if isinstance(payload, dict) else type(payload)}"
    )


def checkpoint_matrix(path, parameter_name):
    state = checkpoint_state_dict(load_checkpoint_payload(path))
    candidates = (
        parameter_name,
        parameter_name.removeprefix("model."),
        f"model.{parameter_name}",
    )
    for name in candidates:
        if name in state:
            value = state[name]
            if hasattr(value, "detach"):
                return value.detach().cpu().double().numpy()
            return np.asarray(value, dtype=np.float64)
    raise KeyError(
        f"Matrix {parameter_name!r} is absent from {path}; "
        f"available 2-D keys={[key for key, value in state.items() if getattr(value, 'ndim', 0) == 2]}"
    )


def checkpoint_step(payload, fallback):
    for key in ("global_step", "step", "optimizer_step"):
        if isinstance(payload, dict) and key in payload:
            return int(payload[key])
    return int(fallback)


def capture_payloads(seed_dir, *, maximum_captures=None):
    capture_root = Path(seed_dir) / "captures"
    require_path(capture_root, description="dense capture directory")
    paths = tuple(list_capture_files(capture_root))
    if not paths:
        raise FileNotFoundError(
            f"No dense capture files under {capture_root}. Ensure the resolved "
            "config includes burst anchors and rerun training."
        )
    if maximum_captures is not None:
        paths = paths[-int(maximum_captures):]
    manifest = json.loads(
        require_path(Path(seed_dir) / "manifest.json", description="run manifest")
        .read_text(encoding="utf-8")
    )
    loaded = []
    for path in paths:
        payload = load_step_capture(
            path, expected_fingerprint=manifest["protocol_fingerprint"]
        )
        if str(payload.get("optimizer")) != str(manifest.get("optimizer")):
            raise RuntimeError(f"Capture optimizer disagrees with manifest: {path}")
        anchor_match = re.fullmatch(r"burst_epoch_(\d+)", path.parent.name)
        if (
            anchor_match is None
            or int(payload.get("anchor_epoch", -1)) != int(anchor_match.group(1))
        ):
            raise RuntimeError(f"Capture anchor epoch disagrees with directory: {path}")
        loaded.append((path, payload))
    return loaded


def capture_array(parameter_payload, key, *, required=True):
    value = parameter_payload.get(key)
    if value is None:
        if required:
            raise KeyError(
                f"Dense capture parameter is missing required field {key!r}; "
                f"available={sorted(parameter_payload)}"
            )
        return None
    if hasattr(value, "detach"):
        return value.detach().cpu().double().numpy()
    return np.asarray(value, dtype=np.float64)


def resolved_training_config(seed_dir):
    payload = json.loads(
        require_path(
            Path(seed_dir) / "resolved_config.json",
            description="resolved training config",
        ).read_text(encoding="utf-8")
    )
    values = dict(payload.get("config", payload))
    if not values:
        raise ValueError(f"Resolved config is empty in {seed_dir}")
    values["adamw"] = AdamWProfile(**dict(values.get("adamw", {})))
    muon_values = dict(values.get("muon", {}))
    if "parameter_names" in muon_values:
        muon_values["parameter_names"] = tuple(muon_values["parameter_names"])
    values["muon"] = MuonProfile(**muon_values)
    clip_values = dict(values.get("muonclip_rms", {}))
    if "parameter_names" in clip_values:
        clip_values["parameter_names"] = tuple(clip_values["parameter_names"])
    values["muonclip_rms"] = MuonClipRMSProfile(**clip_values)
    for name in (
        "explicit_analysis_epochs", "dense_burst_anchor_epochs",
        "capture_parameter_names",
    ):
        if name in values and isinstance(values[name], list):
            values[name] = tuple(values[name])
    config = TangentRGConfig(**values)
    config.validate()
    return config


ECS_COVER_SOURCE_KIND = (
    "verified_tail_checkpoint_cache_plus_exact_sparse_weightwatcher_trace_metrics"
)
ECS_PRIMARY_TRACE_QUALIFICATION_ROLE = "preregistered_independent_fit_support"


def _strict_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    normalized = str(value).strip().lower()
    if normalized in {"true", "1"}:
        return True
    if normalized in {"false", "0"}:
        return False
    raise ValueError(f"Expected a serialized boolean, found {value!r}")


def _strict_integer_metric(value, *, name):
    numeric = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    if pd.isna(numeric):
        raise ValueError(f"{name} is missing or nonnumeric")
    rounded = int(round(float(numeric)))
    if not np.isclose(float(numeric), rounded, rtol=0.0, atol=1.0e-9):
        raise ValueError(f"{name} must be integer-valued, found {value!r}")
    return rounded


def load_verified_ecs_metric_tables(optimizer_slug, seed, expected_fingerprint):
    identity = _VERIFIED_RUN_IDENTITIES.get((str(optimizer_slug), int(seed)))
    if identity is None:
        raise RuntimeError(
            f"Run identity was not verified for optimizer={optimizer_slug}, seed={seed}"
        )
    source_seed_dir = Path(identity["source_seed_dir"]).resolve()
    metrics_dir = source_seed_dir / "metrics"
    fit_path = require_path(
        metrics_dir / "weightwatcher_fits.csv",
        description="WeightWatcher fit table used for exact ECS ranks",
    )
    trace_path = require_path(
        metrics_dir / "trace_log.csv",
        description="trace-log table used to audit exact ECS ranks",
    )
    fits = pd.read_csv(fit_path)
    traces = pd.read_csv(trace_path)
    for label, frame, path, diagnostic_columns in (
        (
            "WeightWatcher",
            fits,
            fit_path,
            {"fit_ok", "detX_num"},
        ),
        (
            "trace-log",
            traces,
            trace_path,
            {
                "qualification_role", "sensitivity_only",
                "certification_eligible", "support_rank_source",
                "support_rank", "support_window_start_descending_zero_based",
                "support_window_end_descending_exclusive",
            },
        ),
    ):
        required = {
            "optimizer", "seed", "protocol_fingerprint", "epoch",
            "global_step", "layer", "fit_variant",
        } | diagnostic_columns
        missing = required - set(frame.columns)
        if missing:
            raise KeyError(f"{path} lacks ECS identity columns {sorted(missing)}")
        identity_columns = [
            "optimizer", "seed", "protocol_fingerprint", "epoch",
            "global_step", "layer", "fit_variant",
        ]
        if frame[identity_columns].isna().any().any():
            raise RuntimeError(f"{label} ECS identity contains nulls in {path}")
        identity_checks = {
            "optimizer": (set(frame["optimizer"].dropna().astype(str)), {str(optimizer_slug)}),
            "seed": (set(frame["seed"].dropna().astype(int)), {int(seed)}),
            "protocol_fingerprint": (
                set(frame["protocol_fingerprint"].dropna().astype(str)),
                {str(expected_fingerprint)},
            ),
        }
        mismatches = [
            f"{field}: observed={sorted(observed)}, expected={sorted(expected)}"
            for field, (observed, expected) in identity_checks.items()
            if observed != expected
        ]
        if mismatches:
            raise RuntimeError(
                f"{label} ECS identity mismatch in {path}: " + "; ".join(mismatches)
            )
    return fits, traces, str(fit_path), str(trace_path)


def exact_ecs_cover_rank_record(
    fits,
    traces,
    *,
    optimizer_slug,
    seed,
    epoch,
    global_step,
    layer,
    maximum_rank,
    fit_path,
    trace_path,
):
    identity_mask_fits = (
        fits["optimizer"].astype(str).eq(str(optimizer_slug))
        & pd.to_numeric(fits["seed"], errors="coerce").eq(int(seed))
        & pd.to_numeric(fits["epoch"], errors="coerce").eq(int(epoch))
        & pd.to_numeric(fits["global_step"], errors="coerce").eq(int(global_step))
        & fits["layer"].astype(str).eq(str(layer))
        & fits["fit_variant"].astype(str).eq("clip_xmax")
    )
    fit_rows = fits.loc[identity_mask_fits]
    base = {
        "ecs_rank_status": "unavailable",
        "ecs_rank_metrics_available": False,
        "ecs_full_shell_available": False,
        "ecs_detx_shell_available": False,
        "ecs_rank_fit_variant": "clip_xmax",
        "ecs_rank_fit_path": str(fit_path),
        "ecs_rank_trace_path": str(trace_path),
        "ecs_rank_exact_match_required": True,
        "ecs_rank_exact_epoch_match_found": False,
        "ecs_rank_exact_global_step_match_found": False,
        "ecs_rank_exact_weightwatcher_state_found": False,
        "ecs_rank_exact_trace_state_found": False,
        "ecs_rank_nearest_or_forward_fill_used": False,
    }
    if fit_rows.empty:
        return {
            **base,
            "ecs_rank_unavailable_reason": "no exact sparse WeightWatcher state",
        }
    if len(fit_rows) != 1:
        raise RuntimeError(
            "Expected one exact clip_xmax WeightWatcher row for "
            f"{optimizer_slug}/seed={seed}/epoch={epoch}/{layer}; found {len(fit_rows)}"
        )
    base.update({
        "ecs_rank_exact_epoch_match_found": True,
        "ecs_rank_exact_global_step_match_found": True,
        "ecs_rank_exact_weightwatcher_state_found": True,
    })
    identity_mask_traces = (
        traces["optimizer"].astype(str).eq(str(optimizer_slug))
        & pd.to_numeric(traces["seed"], errors="coerce").eq(int(seed))
        & pd.to_numeric(traces["epoch"], errors="coerce").eq(int(epoch))
        & pd.to_numeric(traces["global_step"], errors="coerce").eq(int(global_step))
        & traces["layer"].astype(str).eq(str(layer))
        & traces["fit_variant"].astype(str).eq("clip_xmax")
    )
    exact_traces = traces.loc[identity_mask_traces]
    if not exact_traces.empty:
        base["ecs_rank_exact_trace_state_found"] = True
    fit = fit_rows.iloc[0]
    if "fit_ok" not in fit_rows.columns or not _strict_bool(fit["fit_ok"]):
        return {
            **base,
            "ecs_rank_unavailable_reason": "exact WeightWatcher fit is not fit_ok",
            "weightwatcher_status": str(fit.get("weightwatcher_status", fit.get("status", "unknown"))),
        }

    primary = exact_traces.loc[
        exact_traces["qualification_role"].astype(str).eq(
            ECS_PRIMARY_TRACE_QUALIFICATION_ROLE
        )
        & ~exact_traces["sensitivity_only"].map(_strict_bool)
    ]
    if len(primary) != 1:
        if len(primary) > 1:
            raise RuntimeError(
                f"Duplicate primary ECS support rows at epoch={epoch}, layer={layer}"
            )
        return {
            **base,
            "ecs_rank_unavailable_reason": "no exact certifying primary trace support",
        }
    primary = primary.iloc[0]
    if not _strict_bool(primary["certification_eligible"]):
        return {
            **base,
            "ecs_rank_unavailable_reason": "primary trace support is not certification eligible",
        }
    detx_rows = exact_traces.loc[
        exact_traces["support_rank_source"].astype(str).eq("weightwatcher_detX")
    ]
    if len(detx_rows) != 1:
        if len(detx_rows) > 1:
            raise RuntimeError(
                f"Duplicate detX ECS support rows at epoch={epoch}, layer={layer}"
            )
        return {
            **base,
            "ecs_rank_unavailable_reason": "no exact detX trace audit row",
        }
    detx_row = detx_rows.iloc[0]
    midpoint_rows = exact_traces.loc[
        exact_traces["support_rank_source"].astype(str).eq(
            "weightwatcher_midpoint"
        )
    ]
    if len(midpoint_rows) != 1:
        if len(midpoint_rows) > 1:
            raise RuntimeError(
                f"Duplicate midpoint ECS audit rows at epoch={epoch}, layer={layer}"
            )
        return {
            **base,
            "ecs_rank_unavailable_reason": "no exact WeightWatcher midpoint audit row",
        }
    midpoint_row = midpoint_rows.iloc[0]

    detx_value = pd.to_numeric(pd.Series([fit.get("detX_num")]), errors="coerce").iloc[0]
    start_value = pd.to_numeric(
        pd.Series([primary.get("support_window_start_descending_zero_based")]),
        errors="coerce",
    ).iloc[0]
    end_value = pd.to_numeric(
        pd.Series([primary.get("support_window_end_descending_exclusive")]),
        errors="coerce",
    ).iloc[0]
    tail_value = pd.to_numeric(pd.Series([primary.get("support_rank")]), errors="coerce").iloc[0]
    if not all(pd.notna(value) for value in (detx_value, start_value, end_value, tail_value)):
        return {
            **base,
            "ecs_rank_unavailable_reason": "nonfinite PL-window or detX rank",
        }
    try:
        k_pl = _strict_integer_metric(end_value, name="PL support-window end")
        k_tl = _strict_integer_metric(detx_value, name="WeightWatcher detX_num")
        window_start = _strict_integer_metric(
            start_value, name="PL support-window start"
        )
        effective_tail = _strict_integer_metric(
            tail_value, name="PL effective-tail rank"
        )
        detx_trace = _strict_integer_metric(
            detx_row["support_rank"], name="trace detX support rank"
        )
        persisted_midpoint = _strict_integer_metric(
            midpoint_row["support_rank"], name="trace WeightWatcher midpoint rank"
        )
    except ValueError as error:
        raise RuntimeError(
            f"Malformed exact ECS rank metric at epoch={epoch}, layer={layer}: {error}"
        ) from error
    if k_pl != window_start + effective_tail:
        raise RuntimeError("ECS PL boundary is not window_start + effective_tail")
    recorded_pl_boundaries = []
    for name, value in (
        ("WeightWatcher pl_support_rank", fit.get("pl_support_rank")),
        (
            "trace pl_support_rank_before_finger_clip",
            primary.get("pl_support_rank_before_finger_clip"),
        ),
    ):
        numeric = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
        if pd.notna(numeric):
            try:
                recorded_pl_boundaries.append(
                    (name, _strict_integer_metric(numeric, name=name))
                )
            except ValueError as error:
                raise RuntimeError(
                    f"Malformed exact ECS rank metric at epoch={epoch}, "
                    f"layer={layer}: {error}"
                ) from error
    recorded_pl_support_values = {value for _, value in recorded_pl_boundaries}
    if len(recorded_pl_support_values) > 1:
        raise RuntimeError(
            "WeightWatcher and trace recorded PL support counts disagree: "
            + ", ".join(
                f"{name}={value}" for name, value in recorded_pl_boundaries
            )
        )
    recorded_pl_support = (
        next(iter(recorded_pl_support_values))
        if recorded_pl_support_values else None
    )
    if detx_trace != k_tl:
        raise RuntimeError(
            "Exact detX fit field disagrees with trace audit; fallback is refused"
        )
    expected_persisted_midpoint = int(np.floor((effective_tail + k_tl) / 2.0))
    if persisted_midpoint != expected_persisted_midpoint:
        raise RuntimeError(
            "Persisted WeightWatcher midpoint audit disagrees with its declared "
            "effective-tail/detX convention"
        )
    try:
        selection = single_checkpoint.select_ecs_cover_ranks(
            k_pl,
            k_tl,
            maximum_rank=int(maximum_rank),
        )
    except (TypeError, ValueError) as error:
        return {
            **base,
            "ecs_rank_unavailable_reason": f"invalid exact ECS boundaries: {error}",
            "k_pl": k_pl,
            "k_tl": k_tl,
        }
    result = {
        **base,
        "ecs_rank_status": "ok",
        "ecs_rank_metrics_available": True,
        "ecs_full_shell_available": bool(k_pl < int(maximum_rank)),
        "ecs_detx_shell_available": bool(selection.available),
        "ecs_rank_unavailable_reason": "",
        "ecs_full_shell_unavailable_reason": (
            "" if k_pl < int(maximum_rank)
            else "power-law retained boundary reaches checkpoint numerical rank"
        ),
        "ecs_detx_shell_unavailable_reason": selection.unavailable_reason,
        "k_pl": selection.power_law_rank,
        "k_boundary_mid": selection.boundary_midpoint_rank,
        "weightwatcher_midpoint_rank": persisted_midpoint,
        "k_tl": selection.trace_log_rank,
        "retained_rank": selection.retained_rank,
        "retained_rank_source": selection.retained_rank_source,
        "full_shell_outer_rank": int(maximum_rank),
        "full_shell_outer_rank_source": "checkpoint_numerical_rank",
        "full_shell_rank": max(0, int(maximum_rank) - selection.retained_rank),
        "detx_shell_outer_rank": selection.outer_rank,
        "detx_shell_outer_rank_source": selection.outer_rank_source,
        "detx_shell_rank": selection.shell_rank,
        "rank_selection_rule": selection.selection_rule,
        "pl_boundary_definition": (
            "top-mode boundary index = clipped support window start + effective tail rank; "
            "required by V_k=[v_1,...,v_k]; the WeightWatcher-reported PL "
            "support count is audited separately and is not an absolute index"
        ),
        "weightwatcher_pl_support_rank_recorded": recorded_pl_support,
        "pl_effective_tail_rank": effective_tail,
        "pl_support_window_start": window_start,
        "pl_support_window_end": k_pl,
        "pl_support_rank_source": str(primary["support_rank_source"]),
        "pl_support_window_source": str(primary.get("support_window_source", "unknown")),
        "n_fingers_removed": int(round(float(primary.get("n_fingers_removed", window_start)))),
        "detx_rank_source": "finite detX_num in exact WeightWatcher row, cross-audited in trace_log.csv",
    }
    return result


## Exact final-100 checkpoint intersections


In [ ]:
def _ratio_slug(value):
    return str(float(value)).replace("-", "m").replace(".", "p")


operator_records, fit_frames, trace_frames = [], [], []
spectral_arrays = {}
for optimizer in OPTIMIZER_SLUGS:
    for seed in SEEDS:
        seed_dir = require_tail_checkpoint_cache(optimizer, seed)
        run_fingerprint = verified_run_fingerprint(optimizer, seed)
        (
            ecs_metric_fits,
            ecs_metric_traces,
            ecs_fit_path,
            ecs_trace_path,
        ) = load_verified_ecs_metric_tables(
            optimizer, seed, run_fingerprint
        )
        refs = analysis_checkpoint_refs(seed_dir)
        final_epoch = int(refs[-1].epoch)
        for selected, layer, W, selection_rule, selection_role in selected_trajectory_matrices(
            seed_dir,
            layers=LAYERS,
            maximum_checkpoints=MAXIMUM_CHECKPOINTS,
            epoch_stride=ANALYSIS_EPOCH_STRIDE,
        ):
            singular_values = np.linalg.svd(W, compute_uv=False)
            rank_tolerance = float(ECS_RANK_RCOND) * float(singular_values[0])
            numerical_rank = int(
                np.count_nonzero(singular_values > rank_tolerance)
            )
            ranks = exact_ecs_cover_rank_record(
                ecs_metric_fits,
                ecs_metric_traces,
                optimizer_slug=optimizer,
                seed=seed,
                epoch=int(selected.epoch),
                global_step=int(selected.global_step),
                layer=layer,
                maximum_rank=numerical_rank,
                fit_path=ecs_fit_path,
                trace_path=ecs_trace_path,
            )
            common = {
                "optimizer": optimizer,
                "seed": int(seed),
                "layer": layer,
                "protocol_fingerprint": run_fingerprint,
                "source_artifact_kind": ECS_COVER_SOURCE_KIND,
                "state_index": int(selected.global_step),
                "epoch": int(selected.epoch),
                "trajectory_selection_rule": selection_rule,
                "trajectory_selection_role": selection_role,
                "checkpoint_source": "verified_final_100_tail_cache",
                "checkpoint_cache_seed_dir": str(seed_dir),
                "analysis_contract_token": ANALYSIS_CONTRACT_TOKEN,
                "checkpoint_numerical_rank": numerical_rank,
                "checkpoint_rank_tolerance": rank_tolerance,
                **ranks,
            }
            candidates = []
            if not bool(ranks["ecs_full_shell_available"]):
                operator_records.append({
                    **common,
                    "method": "additional_weight_only_ecs_jacobians_unavailable",
                    "operator_kind": "unavailable_exact_sparse_ecs_rank_state",
                    "map_definition": "all five requested maps require an exact same-state retained ECS boundary",
                    "available": False,
                    "unavailable_reason": ranks.get(
                        "ecs_rank_unavailable_reason",
                        ranks.get("full_shell_unavailable_reason", "unknown"),
                    ),
                })
                continue

            k = int(ranks["retained_rank"])
            shell_policies = [
                (
                    "full_row_shell",
                    int(ranks["full_shell_outer_rank"]),
                    "primary_full_checkpoint_numerical_row_shell",
                )
            ]
            if bool(ranks["ecs_detx_shell_available"]):
                shell_policies.append((
                    "detx_shell",
                    int(ranks["detx_shell_outer_rank"]),
                    "detx_bounded_shell_sensitivity_only",
                ))
            else:
                operator_records.append({
                    **common,
                    "method": "detx_shell_variants_unavailable",
                    "operator_kind": "unavailable_detx_bounded_shell_sensitivity",
                    "map_definition": "detX-q variants require k<q_detX; full-q primary remains available",
                    "available": False,
                    "unavailable_reason": ranks.get(
                        "detx_shell_unavailable_reason", "detX shell unavailable"
                    ),
                })

            for shell_name, q, shell_role in shell_policies:
                shell_base = {
                    **common,
                    "retained_rank": k,
                    "outer_rank": q,
                    "shell_rank": q - k,
                    "ecs_shell_variant": shell_name,
                    "ecs_shell_selection_role": shell_role,
                }
                gap = ecs_jacobians.gap_aware_projector_spectrum(
                    W,
                    retained_rank=k,
                    outer_rank=q,
                    rcond=ECS_RANK_RCOND,
                    precomputed_singular_values=singular_values,
                )
                candidates.append((
                    f"gap_aware_grassmann_projector_{shell_name}",
                    "gap_aware_grassmann_projector",
                    gap,
                    shell_base,
                ))
                log_gram = ecs_jacobians.outer_trace_free_log_gram_spectrum(
                    W,
                    outer_rank=q,
                    rcond=ECS_RANK_RCOND,
                    precomputed_singular_values=singular_values,
                )
                candidates.append((
                    f"exact_trace_free_log_gram_{shell_name}",
                    "exact_trace_free_log_gram",
                    log_gram,
                    shell_base,
                ))
                feshbach_z = (
                    float(FESHBACH_Z_SHELL_FLOOR_RATIO)
                    * float(singular_values[q - 1] ** 2)
                )
                feshbach = ecs_jacobians.feshbach_trace_free_log_spectrum(
                    W,
                    retained_rank=k,
                    outer_rank=q,
                    z=feshbach_z,
                    rcond=ECS_RANK_RCOND,
                )
                candidates.append((
                    f"feshbach_trace_free_log_core_{shell_name}",
                    "feshbach_trace_free_log_effective_core",
                    feshbach,
                    {
                        **shell_base,
                        "feshbach_z": feshbach_z,
                        "first_order_shell_downfolding_active": False,
                        "feshbach_base_coupling_norm": feshbach.parameters[
                            "base_coupling_norm"
                        ],
                    },
                ))
                boundary_scale = float(singular_values[k - 1] ** 2)
                for ratio in RESOLVENT_Z_BOUNDARY_RATIOS:
                    z = float(ratio) * boundary_scale
                    resolvent = ecs_jacobians.outer_resolvent_spectrum(
                        W,
                        outer_rank=q,
                        z=z,
                        trace_free=True,
                        rcond=ECS_RANK_RCOND,
                        precomputed_singular_values=singular_values,
                    )
                    candidates.append((
                        f"trace_free_resolvent_{shell_name}_zratio_{_ratio_slug(ratio)}",
                        "trace_free_resolvent",
                        resolvent,
                        {
                            **shell_base,
                            "resolvent_z": z,
                            "resolvent_z_boundary_ratio": float(ratio),
                            "resolvent_boundary_scale": boundary_scale,
                        },
                    ))

            squared = singular_values**2
            boundary_gap = float(squared[k - 1] - squared[k])
            lambda_center = float(0.5 * (squared[k - 1] + squared[k]))
            for ratio in SOFT_TEMPERATURE_GAP_RATIOS:
                temperature = max(
                    float(ratio) * boundary_gap,
                    np.finfo(float).eps * float(squared[0]),
                )
                soft = ecs_jacobians.soft_ecs_projector_spectrum(
                    W,
                    lambda_center=lambda_center,
                    temperature=temperature,
                    rcond=ECS_RANK_RCOND,
                    precomputed_singular_values=singular_values,
                )
                candidates.append((
                    f"soft_ecs_projector_taugap_{_ratio_slug(ratio)}",
                    "soft_ecs_projector",
                    soft,
                    {
                        **common,
                        "retained_rank": k,
                        "outer_rank": numerical_rank,
                        "ecs_shell_variant": "full_right_gram_soft_boundary",
                        "soft_lambda_center": lambda_center,
                        "soft_boundary_gap": boundary_gap,
                        "soft_temperature": temperature,
                        "soft_temperature_gap_ratio": float(ratio),
                    },
                ))

            for method, family, record, candidate_base in candidates:
                amplitudes = np.asarray(
                    record.singular_amplitudes, dtype=float
                )
                amplitudes = amplitudes[
                    np.isfinite(amplitudes) & (amplitudes > 0.0)
                ]
                operator_row = {
                    **candidate_base,
                    "method": method,
                    "jacobian_family": family,
                    "operator_kind": record.operator_kind,
                    "map_definition": record.map_definition,
                    "derivative_rank": int(record.derivative_rank),
                    "input_dimension": int(record.input_dimension),
                    "output_dimension": int(record.output_dimension),
                    "zero_count": int(record.zero_count),
                    "available": bool(amplitudes.size >= 2),
                    "energy_convention": "nonzero_eigenvalues_of_J_star_J",
                    "jacobian_parameters": json.dumps(
                        dict(record.parameters), sort_keys=True
                    ),
                }
                operator_records.append(operator_row)
                if amplitudes.size < 2:
                    continue
                fits, traces = fit_spectrum_with_trace(
                    amplitudes,
                    operator_kind=record.operator_kind,
                    map_definition=record.map_definition,
                    spectrum_kind="amplitude",
                    metadata={
                        **candidate_base,
                        "method": method,
                        "jacobian_family": family,
                        "energy_convention": "nonzero_eigenvalues_of_J_star_J",
                    },
                    top_k_values=TOP_K_VALUES,
                    minimum_tail=MINIMUM_TAIL,
                )
                fit_frames.append(fits)
                trace_frames.append(traces)
                if int(selected.epoch) == final_epoch:
                    spectral_arrays[
                        f"{optimizer}_{seed}_{layer}_{selected.global_step}_{method}"
                    ] = amplitudes

# Independent small-matrix materialization of every declared map.
small = np.zeros((4, 6), dtype=float)
small[:4, :4] = np.diag([5.0, 3.0, 2.0, 1.0])
validation_cases = (
    (
        "gap_aware_grassmann_projector",
        lambda candidate: ecs_jacobians.gap_aware_projector_map(
            small, candidate, retained_rank=2, outer_rank=4, rcond=1e-12
        ).value,
        ecs_jacobians.gap_aware_projector_spectrum(
            small, retained_rank=2, outer_rank=4, rcond=1e-12
        ),
    ),
    (
        "soft_ecs_projector",
        lambda candidate: ecs_jacobians.soft_ecs_projector_map(
            candidate, lambda_center=6.5, temperature=10.0
        ).value,
        ecs_jacobians.soft_ecs_projector_spectrum(
            small, lambda_center=6.5, temperature=10.0, rcond=1e-12
        ),
    ),
    (
        "exact_trace_free_log_gram",
        lambda candidate: ecs_jacobians.outer_trace_free_log_gram_map(
            small, candidate, outer_rank=4, rcond=1e-12
        ).value,
        ecs_jacobians.outer_trace_free_log_gram_spectrum(
            small, outer_rank=4, rcond=1e-12
        ),
    ),
    (
        "trace_free_resolvent",
        lambda candidate: ecs_jacobians.outer_resolvent_map(
            small, candidate, outer_rank=4, z=2.0,
            trace_free=True, rcond=1e-12
        ).value,
        ecs_jacobians.outer_resolvent_spectrum(
            small, outer_rank=4, z=2.0,
            trace_free=True, rcond=1e-12
        ),
    ),
    (
        "feshbach_trace_free_log_effective_core",
        lambda candidate: ecs_jacobians.feshbach_trace_free_log_map(
            small, candidate, retained_rank=2, outer_rank=4,
            z=0.5, rcond=1e-12
        ).value,
        ecs_jacobians.feshbach_trace_free_log_spectrum(
            small, retained_rank=2, outer_rank=4,
            z=0.5, rcond=1e-12
        ),
    ),
)
for name, map_function, analytic in validation_cases:
    numerical = polar.central_difference_jacobian(
        map_function,
        small,
        max_input_dimension=small.size,
        rank_rtol=1e-8,
        operator_kind=f"explicit_numerical_{name}_jacobian",
        map_definition=f"central-difference materialization of {name}",
    )
    expected = np.asarray(analytic.singular_amplitudes, dtype=float)
    observed = numerical.singular_values[: expected.size]
    relative_error = float(
        np.linalg.norm(observed - expected)
        / max(np.linalg.norm(expected), np.finfo(float).tiny)
    )
    operator_records.append({
        "optimizer": "synthetic",
        "seed": int(SEEDS[0]),
        "layer": str(tuple(small.shape)),
        "protocol_fingerprint": "not_applicable_synthetic_formula_validation",
        "source_artifact_kind": "fixed_seed_synthetic_formula_validation",
        "state_index": 0,
        "method": f"small_explicit_{name}_validation",
        "jacobian_family": name,
        "operator_kind": numerical.operator_kind,
        "map_definition": numerical.map_definition,
        "analytic_operator_kind": analytic.operator_kind,
        "numeric_rank": int(numerical.numerical_rank),
        "analytic_rank": int(analytic.derivative_rank),
        "relative_spectral_error": relative_error,
        "available": True,
    })
    if numerical.numerical_rank != analytic.derivative_rank:
        raise RuntimeError(
            f"{name} numerical rank {numerical.numerical_rank} "
            f"!= analytic rank {analytic.derivative_rank}"
        )
    if relative_error > 2e-4:
        raise RuntimeError(
            f"{name} analytic/numerical spectrum mismatch {relative_error:.3e}"
        )
if not fit_frames:
    raise RuntimeError("No additional weight-only Jacobian spectra were fit")


In [ ]:
operator_rows = pd.DataFrame(operator_records)
fit_rows = pd.concat(fit_frames, ignore_index=True, sort=False)
trace_rows = pd.concat(trace_frames, ignore_index=True, sort=False)
required_provenance = {"operator_kind", "map_definition"}
for name, frame in {
    "operator rows": operator_rows,
    "fit rows": fit_rows,
    "trace rows": trace_rows,
}.items():
    missing = required_provenance - set(frame.columns)
    if missing:
        raise RuntimeError(f"{name} missing provenance columns: {sorted(missing)}")
    if frame[list(required_provenance)].isna().any().any():
        raise RuntimeError(f"{name} has null provenance")
analysis_dir = save_analysis_frames(
    METHOD_SLUG,
    operators=operator_rows,
    fits=fit_rows,
    traces=trace_rows,
)
alpha_summary, _ = plot_fit_alpha_ci(
    fit_rows,
    method_slug=METHOD_SLUG,
    title=PLOT_TITLE,
)
alpha_summary.to_csv(analysis_dir / "alpha_summary_95ci.csv", index=False)
display(operator_rows.head(24))
display(fit_rows.head(24))
display(trace_rows.head(24))
print("analysis outputs:", analysis_dir)


In [ ]:
np.savez_compressed(analysis_dir / "positive_spectra.npz", **spectral_arrays)
display(save_spectrum_ccdf_gallery(spectral_arrays, method_slug=METHOD_SLUG))
validation_rows = operator_rows[
    operator_rows["method"].astype(str).str.startswith("small_explicit_")
].copy()
validation_rows.to_csv(
    analysis_dir / "small_explicit_jacobian_validation.csv", index=False
)
availability = (
    operator_rows[
        ~operator_rows["optimizer"].astype(str).eq("synthetic")
    ]
    .groupby(["optimizer", "seed", "jacobian_family"], dropna=False)
    .agg(
        attempted_states=("state_index", "nunique"),
        available_rows=("available", lambda values: int(boolean_series(values).sum())),
    )
    .reset_index()
)
availability.to_csv(
    analysis_dir / "jacobian_availability_by_run.csv", index=False
)
display(validation_rows)
display(availability)


`soft_ecs_projector` is differentiated on the full right Gram,
including active-to-null modes. Resolvent rows sweep a fixed
preregistered set of $z/sigma_k^2$ scales. The full-$q$ rows are
primary geometric definitions; detX-$q$ rows are shell-boundary
sensitivities. The Feshbach result must be interpreted literally:
in the SVD gauge $B=0$, hence shell downfolding first appears beyond
linear order and its Jacobian equals the retained trace-free
log-core response.
